## 1. Cấu hình kết nối

Code đọc `.env` thay vì hard-code password. `ROOT` xử lý cả hai trường hợp: mở Jupyter tại project root hoặc từ một subfolder. URI host là `bolt://localhost:7688` vì Docker map Neo4j Bolt container port 7687 ra port 7688 của máy.

In [8]:
from __future__ import annotations

import json
import os
from collections import Counter
from pathlib import Path

from dotenv import load_dotenv
from neo4j import GraphDatabase

ROOT = Path.cwd().resolve()
if not (ROOT / 'data' / 'course_graph.json').exists():
    ROOT = ROOT.parent

load_dotenv(override=True)
NEO4J_URI = os.getenv('NEO4J_URI', 'bolt://localhost:7688')
NEO4J_USERNAME = os.getenv('NEO4J_USERNAME', 'neo4j')
NEO4J_PASSWORD = os.getenv('NEO4J_PASSWORD', '').strip()
NEO4J_DATABASE = os.getenv('NEO4J_DATABASE', 'neo4j')
# if not NEO4J_PASSWORD or 'change-this' in NEO4J_PASSWORD:
#     raise ValueError('Hãy tạo .env từ .env.example và đặt NEO4J_PASSWORD trước.')

JSON_PATH = ROOT / 'data' / 'course_graph.json'
print({'root': str(ROOT), 'json_path': str(JSON_PATH), 'neo4j_uri': NEO4J_URI, 'database': NEO4J_DATABASE})


{'root': '/Users/vunguyen/Documents/Mine/Python_Cyber/Lesson12', 'json_path': '/Users/vunguyen/Documents/Mine/Python_Cyber/Lesson12/data/course_graph.json', 'neo4j_uri': 'bolt://localhost:7688', 'database': 'neo4j'}


## 2. Đọc raw JSON và validate trước khi ghi database

JSON tách `nodes` và `relationships`. `ref` là identity kỹ thuật ổn định cho import; property domain như `id`, `dataset_id + key`, `chunk_id` vẫn giữ nguyên để query. Validation làm ba việc: cấm label/type ngoài whitelist trước khi chúng được đưa vào Cypher động, phát hiện ref trùng, và đảm bảo mọi relationship trỏ tới node có thật. Đây là nguyên tắc quan trọng khi import JSON bên ngoài.

In [9]:
with JSON_PATH.open(encoding='utf-8') as file:
    graph_data = json.load(file)

ALLOWED_LABELS = {'Practice', 'Learner', 'Course', 'Skill', 'Role', 'Topic', 'Document', 'Chunk', 'Entity'}
ALLOWED_RELATIONSHIP_TYPES = {
    'INTERESTED_IN', 'HAS_SKILL', 'TEACHES', 'COVERS', 'LEADS_TO', 'REQUIRES',
    'BUILT_WITH', 'CONTAINS', 'MENTIONS', 'RELATED',
}

nodes = graph_data['nodes']
relationships = graph_data['relationships']
node_refs = [node['ref'] for node in nodes]
assert len(node_refs) == len(set(node_refs)), 'Node ref bị trùng'
assert all(set(node['labels']) <= ALLOWED_LABELS for node in nodes), 'Có label không được phép'
assert all('Practice' in node['labels'] for node in nodes), 'Mỗi node phải có label Practice'
assert all(rel['type'] in ALLOWED_RELATIONSHIP_TYPES for rel in relationships), 'Có relationship type không được phép'
assert all(rel['from_ref'] in node_refs and rel['to_ref'] in node_refs for rel in relationships), 'Relationship tham chiếu node không tồn tại'

print({
    'dataset_id': graph_data['dataset_id'],
    'nodes': len(nodes),
    'relationships': len(relationships),
    'labels': Counter(label for node in nodes for label in node['labels']),
    'relationship_types': Counter(rel['type'] for rel in relationships),
})


{'dataset_id': 'course-catalog', 'nodes': 38, 'relationships': 59, 'labels': Counter({'Practice': 38, 'Entity': 12, 'Skill': 7, 'Role': 4, 'Chunk': 4, 'Learner': 3, 'Course': 3, 'Topic': 3, 'Document': 2}), 'relationship_types': Counter({'MENTIONS': 16, 'RELATED': 14, 'TEACHES': 7, 'LEADS_TO': 5, 'COVERS': 4, 'CONTAINS': 4, 'INTERESTED_IN': 3, 'HAS_SKILL': 3, 'REQUIRES': 2, 'BUILT_WITH': 1})}


## 3. Kết nối và tạo constraint

`verify_connectivity()` fail sớm nếu Docker chưa chạy hoặc password sai. Constraint `practice_node_ref` đảm bảo raw JSON import idempotent: chạy lại `MERGE` với cùng `ref` sẽ update node cũ thay vì tạo node mới. Constraint `(dataset_id, key)` là identity gần với Graph RAG thật; cùng entity key được phép tồn tại ở dataset khác nhưng không được trùng trong một dataset.

In [10]:
driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USERNAME, NEO4J_PASSWORD))
driver.verify_connectivity()

CONSTRAINTS = [
    'CREATE CONSTRAINT practice_node_ref IF NOT EXISTS FOR (n:Practice) REQUIRE n.practice_ref IS UNIQUE',
    'CREATE CONSTRAINT practice_entity_identity IF NOT EXISTS FOR (e:Entity) REQUIRE (e.dataset_id, e.key) IS UNIQUE',
]

with driver.session(database=NEO4J_DATABASE) as session:
    for statement in CONSTRAINTS:
        session.run(statement).consume()

print('Neo4j connection and constraints are ready.')


Neo4j connection and constraints are ready.


## 4. Load JSON vào Graph Database

Cell này cố ý biểu diễn importer rõ ràng thay vì che trong framework. Node được `MERGE` bằng `:Practice {practice_ref}`, sau đó `SET n += $properties` ghi property từ JSON. Labels và relationship types không parameterize được trong Cypher, vì vậy code **validate whitelist trước** rồi mới ghép chúng vào query. Relationship cũng `MERGE` theo `practice_ref` để re-run không duplicate edge.

`RESET_PRACTICE_GRAPH = True` chỉ xóa graph mang label `:Practice` của database bài tập rồi nạp raw JSON sạch. Giữ `False` nếu bạn muốn tự quan sát tính idempotent của `MERGE`.

In [11]:
RESET_PRACTICE_GRAPH = True

def reset_practice_graph(tx):
    tx.run('MATCH (n:Practice) DETACH DELETE n').consume()

def upsert_node(tx, node):
    properties = {**node['properties'], 'practice_ref': node['ref']}
    tx.run(
        'MERGE (n:Practice {practice_ref: $ref}) SET n += $properties',
        ref=node['ref'],
        properties=properties,
    ).consume()
    for label in set(node['labels']) - {'Practice'}:
        # label đã được validate bằng ALLOWED_LABELS, nên interpolation này an toàn.
        tx.run(f'MATCH (n:Practice {{practice_ref: $ref}}) SET n:`{label}`', ref=node['ref']).consume()

def upsert_relationship(tx, relationship):
    rel_type = relationship['type']  # Đã validate bằng ALLOWED_RELATIONSHIP_TYPES.
    properties = {**relationship['properties'], 'practice_ref': relationship['ref']}
    tx.run(
        f'''
        MATCH (source:Practice {{practice_ref: $from_ref}})
        MATCH (target:Practice {{practice_ref: $to_ref}})
        MERGE (source)-[r:`{rel_type}` {{practice_ref: $ref}}]->(target)
        SET r += $properties
        ''',
        from_ref=relationship['from_ref'],
        to_ref=relationship['to_ref'],
        ref=relationship['ref'],
        properties=properties,
    ).consume()

with driver.session(database=NEO4J_DATABASE) as session:
    if RESET_PRACTICE_GRAPH:
        session.execute_write(reset_practice_graph)
    for node in nodes:
        session.execute_write(upsert_node, node)
    for relationship in relationships:
        session.execute_write(upsert_relationship, relationship)

print(f'Imported {len(nodes)} nodes and {len(relationships)} relationships from JSON.')


Imported 38 nodes and 59 relationships from JSON.


## 5. Helper chạy Cypher có parameter

Helper trả `record.data()` để Jupyter hiển thị list dictionary dễ đọc. Mọi giá trị động như `dataset_id`, `seed_names`, `max_facts` đi qua `$parameter`; chỉ hop count được xử lý riêng ở cell Graph RAG vì syntax `*1..N` của Cypher không parameterize được.

In [12]:
def run_cypher(query: str, **parameters) -> list[dict]:
    with driver.session(database=NEO4J_DATABASE) as session:
        return [record.data() for record in session.run(query, **parameters)]

run_cypher(
    'MATCH (n:Practice) RETURN labels(n) AS labels, count(*) AS count ORDER BY labels'
)


[{'labels': ['Practice', 'Chunk'], 'count': 4},
 {'labels': ['Practice', 'Course'], 'count': 3},
 {'labels': ['Practice', 'Document'], 'count': 2},
 {'labels': ['Practice', 'Entity'], 'count': 12},
 {'labels': ['Practice', 'Learner'], 'count': 3},
 {'labels': ['Practice', 'Role'], 'count': 4},
 {'labels': ['Practice', 'Skill'], 'count': 7},
 {'labels': ['Practice', 'Topic'], 'count': 3}]

## Solutions F01–F06 — Sales, CRM lead và entity catalogue

Một dictionary gom các response để bạn nhìn đúng output contract của từng ticket. `run_cypher` vẫn gửi mọi dynamic value qua parameter.

In [13]:
# F01 — Course catalogue cho sales
run_cypher('''
MATCH (course:Course:Practice)
RETURN course.title as title, course.level as level, course.duration_weeks as duration_weeks, course.tuition_vnd as tuition_vnd
ORDER BY course.duration_weeks ASC, course.title
''')


[{'title': 'AI and Data Science',
  'level': 'Intermediate',
  'duration_weeks': 12,
  'tuition_vnd': 10150000},
 {'title': 'Data Analytics Foundations',
  'level': 'Beginner',
  'duration_weeks': 16,
  'tuition_vnd': 8500000},
 {'title': 'Java Backend',
  'level': 'Beginner',
  'duration_weeks': 24,
  'tuition_vnd': 12000000}]

In [15]:
# F02 — Lead có mục tiêu liên quan data
run_cypher('''
MATCH (learner:Learner:Practice)
WHERE toLower(learner.goal) CONTAINS toLower($goal_keyword)
RETURN learner.name as name, learner.background as background, learner.goal as goal

''', goal_keyword="data")


[{'name': 'Lan',
  'background': 'Office operations',
  'goal': 'Become a data analyst'}]

In [19]:
# F03 — Entity catalogue cho chatbot
# run_cypher('''
# MATCH(entity:Entity)
# RETURN entity
# ''')


run_cypher('''
MATCH (entity:Entity)
RETURN entity.dataset_id, entity.key, entity.name, entity.type, entity.description

''')

[{'entity.dataset_id': 'course-catalog',
  'entity.key': 'data analytics foundations',
  'entity.name': 'Data Analytics Foundations',
  'entity.type': 'Course',
  'entity.description': 'A 16-week beginner data course'},
 {'entity.dataset_id': 'course-catalog',
  'entity.key': 'ai and data science',
  'entity.name': 'AI and Data Science',
  'entity.type': 'Course',
  'entity.description': 'A 12-week intermediate AI course'},
 {'entity.dataset_id': 'course-catalog',
  'entity.key': 'python',
  'entity.name': 'Python',
  'entity.type': 'Skill',
  'entity.description': 'Programming language for data work'},
 {'entity.dataset_id': 'course-catalog',
  'entity.key': 'sql',
  'entity.name': 'SQL',
  'entity.type': 'Skill',
  'entity.description': 'Language for querying data'},
 {'entity.dataset_id': 'course-catalog',
  'entity.key': 'power bi',
  'entity.name': 'Power BI',
  'entity.type': 'Skill',
  'entity.description': 'Business intelligence tool'},
 {'entity.dataset_id': 'course-catalog',


In [21]:
# F04 — Source chunks cho QA audit
run_cypher('''
MATCH (chunk:Chunk:Practice)
return *
''')


[{'chunk': {'practice_ref': 'chunk:analytics-01',
   'dataset_id': 'course-catalog',
   'index': 1,
   'text': 'Data Analytics Foundations is a 16-week beginner course. Students learn Python, SQL, Power BI and ETL for data analysis.',
   'document_id': 'analytics-brochure',
   'title': 'Data Analytics brochure',
   'chunk_id': 'analytics-01'}},
 {'chunk': {'practice_ref': 'chunk:analytics-02',
   'dataset_id': 'course-catalog',
   'index': 2,
   'text': 'Graduates can prepare for Data Analyst and Data Scientist roles. The course includes forecasting projects.',
   'document_id': 'analytics-brochure',
   'title': 'Data Analytics brochure',
   'chunk_id': 'analytics-02'}},
 {'chunk': {'practice_ref': 'chunk:ai-01',
   'dataset_id': 'course-catalog',
   'index': 1,
   'text': 'AI and Data Science is a 12-week intermediate course. It requires Python and SQL, then teaches Machine Learning and Deep Learning.',
   'document_id': 'ai-brochure',
   'title': 'AI and Data Science brochure',
   'c

In [ ]:
# F05 — Query parameterized: đổi dataset_id không phải sửa query string
run_cypher('''

''', dataset_id=)


[{'name': 'AI and Data Science'},
 {'name': 'Data Analyst'},
 {'name': 'Data Analytics Foundations'},
 {'name': 'Data Scientist'},
 {'name': 'Deep Learning'},
 {'name': 'Forecasting'},
 {'name': 'Graph RAG'},
 {'name': 'Machine Learning'},
 {'name': 'Machine Learning Engineer'},
 {'name': 'Power BI'},
 {'name': 'Python'},
 {'name': 'SQL'}]

In [ ]:
# F06 — Entity card contract cho frontend
run_cypher('''

''', dataset_id='course-catalog')


[{'entity_name': 'AI and Data Science',
  'entity_type': 'Course',
  'summary': 'A 12-week intermediate AI course'},
 {'entity_name': 'Data Analyst',
  'entity_type': 'Role',
  'summary': 'Data analysis role'},
 {'entity_name': 'Data Analytics Foundations',
  'entity_type': 'Course',
  'summary': 'A 16-week beginner data course'},
 {'entity_name': 'Data Scientist',
  'entity_type': 'Role',
  'summary': 'Data science role'},
 {'entity_name': 'Deep Learning',
  'entity_type': 'Topic',
  'summary': 'Neural-network modelling'},
 {'entity_name': 'Forecasting',
  'entity_type': 'Topic',
  'summary': 'Predicting future values'},
 {'entity_name': 'Graph RAG',
  'entity_type': 'Topic',
  'summary': 'Graph-based retrieval augmented generation'},
 {'entity_name': 'Machine Learning',
  'entity_type': 'Topic',
  'summary': 'Predictive modelling'},
 {'entity_name': 'Machine Learning Engineer',
  'entity_type': 'Role',
  'summary': 'Machine learning engineering role'},
 {'entity_name': 'Power BI',
  

## Solutions R01–R10 — tư vấn lộ trình và curriculum dashboard

Các query này cho thấy cùng một graph đáp ứng nhiều màn hình nghiệp vụ: tín hiệu lead, curriculum theo module, career outcome, dashboard count và fact shape cho LLM. Solution R04 đã nằm ở cell trước để bạn thấy path 2-hop độc lập.

In [24]:
# R01 — Tín hiệu quan tâm của Lan
run_cypher('''
MATCH(learner:Learner {name: $name})-[r:INTERESTED_IN] -> (course:Course)
return learner.name as learner, type(r) as relationships, course.title as course
''', name="Lan")


[{'learner': 'Lan',
  'relationships': 'INTERESTED_IN',
  'course': 'Data Analytics Foundations'}]

In [29]:
# R02 — Skills và module của Data Analytics Foundations
run_cypher('''
MATCH(course:Course {title: $title})-[r:TEACHES] -> (skill:Skill)
return skill.name as skill, r.module as module
''', title="Data Analytics Foundations")


[{'skill': 'Power BI', 'module': 3},
 {'skill': 'SQL', 'module': 2},
 {'skill': 'Python', 'module': 1}]

In [33]:
# R03 — Career outcomes theo Course
run_cypher('''
MATCH(course:Course {title: $title})-[r:LEADS_TO] -> (role:Role)
return role.name as role
''', title="Data Analytics Foundations")



[{'role': 'Data Scientist'}, {'role': 'Data Analyst'}]

In [42]:
# R04 — Path Learner → Course → Skill của An
run_cypher('''
MATCH(learner:Learner {name: $name})-[r1:INTERESTED_IN]->(course:Course)-[r2:TEACHES]->(skill:Skill)
return course.title as course, skill.name AS skill
''', name="An")


[{'course': 'AI and Data Science', 'skill': 'Deep Learning'},
 {'course': 'AI and Data Science', 'skill': 'Machine Learning'}]

In [43]:
# R05 — Entity reachable trong 1–2 RELATED hops
run_cypher('''
MATCH (seed:Entity {name: $seed_name})
MATCH path=(seed)-[r:RELATED*1..2]-(other:Entity) 
return other.name as entity, length(path) as paths ORDER BY path
''',seed_name="AI and Data Science")


[{'entity': 'Python', 'paths': 1},
 {'entity': 'Data Analytics Foundations', 'paths': 2},
 {'entity': 'SQL', 'paths': 1},
 {'entity': 'Data Analytics Foundations', 'paths': 2},
 {'entity': 'Machine Learning', 'paths': 1},
 {'entity': 'Deep Learning', 'paths': 1},
 {'entity': 'Forecasting', 'paths': 1},
 {'entity': 'Data Analytics Foundations', 'paths': 2},
 {'entity': 'Graph RAG', 'paths': 1},
 {'entity': 'Data Scientist', 'paths': 1},
 {'entity': 'Data Analytics Foundations', 'paths': 2},
 {'entity': 'Machine Learning Engineer', 'paths': 1}]

In [ ]:
# R06 — So sánh bỏ/có DISTINCT để thấy duplicate path
{
    'without_distinct':
        run_cypher('''
                    ''',
                      dataset_id='course-catalog',
                      seed_name='AI and Data Science'
                  ),
    'with_distinct':
        run_cypher('''
                    ''',
                      dataset_id='course-catalog',
                      seed_name='AI and Data Science'
                  )
}


{'without_distinct': [{'entity': 'Data Analyst'},
  {'entity': 'Data Analyst'},
  {'entity': 'Data Analyst'},
  {'entity': 'Data Analyst'},
  {'entity': 'Data Analytics Foundations'},
  {'entity': 'Data Analytics Foundations'},
  {'entity': 'Data Analytics Foundations'},
  {'entity': 'Data Analytics Foundations'},
  {'entity': 'Data Scientist'},
  {'entity': 'Data Scientist'},
  {'entity': 'Data Scientist'},
  {'entity': 'Data Scientist'},
  {'entity': 'Deep Learning'},
  {'entity': 'Forecasting'},
  {'entity': 'Forecasting'},
  {'entity': 'Forecasting'},
  {'entity': 'Forecasting'},
  {'entity': 'Graph RAG'},
  {'entity': 'Machine Learning'},
  {'entity': 'Machine Learning Engineer'},
  {'entity': 'Power BI'},
  {'entity': 'Power BI'},
  {'entity': 'Power BI'},
  {'entity': 'Power BI'},
  {'entity': 'Python'},
  {'entity': 'Python'},
  {'entity': 'Python'},
  {'entity': 'Python'},
  {'entity': 'SQL'},
  {'entity': 'SQL'},
  {'entity': 'SQL'},
  {'entity': 'SQL'}],
 'with_distinct': [{

In [ ]:
# R07 — Dashboard số skill, giữ cả Course có count 0
run_cypher('''
        ''')


[{'course': 'Data Analytics Foundations', 'taught_skills': 3},
 {'course': 'AI and Data Science', 'taught_skills': 2},
 {'course': 'Java Backend', 'taught_skills': 2}]

In [ ]:
# R08 — Course có curriculum ít nhất hai skill
run_cypher('''
''', minimum_skills=)


[{'course': 'Data Analytics Foundations', 'taught_skills': 3},
 {'course': 'AI and Data Science', 'taught_skills': 2},
 {'course': 'Java Backend', 'taught_skills': 2}]

In [ ]:
# R09 — Role chưa có Course dẫn tới
run_cypher('''
''')


[]

In [ ]:
# R10 — Fact shape cho LLM context
run_cypher('''
''', dataset_id=, seed_name=)


[{'source': 'AI and Data Science',
  'relationship': 'COVERS',
  'target': 'Forecasting',
  'evidence': 'The course covers forecasting and Graph RAG.',
  'chunk_id': 'ai-02'},
 {'source': 'AI and Data Science',
  'relationship': 'COVERS',
  'target': 'Graph RAG',
  'evidence': 'The course covers forecasting and Graph RAG.',
  'chunk_id': 'ai-02'},
 {'source': 'AI and Data Science',
  'relationship': 'LEADS_TO',
  'target': 'Data Scientist',
  'evidence': 'It prepares learners for Data Scientist and Machine Learning Engineer roles.',
  'chunk_id': 'ai-02'},
 {'source': 'AI and Data Science',
  'relationship': 'LEADS_TO',
  'target': 'Machine Learning Engineer',
  'evidence': 'It prepares learners for Data Scientist and Machine Learning Engineer roles.',
  'chunk_id': 'ai-02'},
 {'source': 'AI and Data Science',
  'relationship': 'REQUIRES',
  'target': 'Python',
  'evidence': 'It requires Python and SQL.',
  'chunk_id': 'ai-01'},
 {'source': 'AI and Data Science',
  'relationship': 'REQ

## Solutions W01–W08 — CRM webhook và vận hành dữ liệu

Tất cả code solution W01–W08 đều ở section này. W01 và W05 được đặt cờ chạy rõ ràng vì chúng tạo/xóa test lead; W02–W04 có thể re-run để tự kiểm tra idempotency. Hoa sẽ tồn tại trong practice graph đến khi bạn chạy cleanup hoặc chạy lại JSON import.

In [ ]:
# W01 — CREATE demo. Đặt True và chạy hai lần để quan sát duplicate do webhook retry.
run_w01_create_demo = False
if run_w01_create_demo:
    run_cypher('''CREATE (learner:Learner:Practice {id: $id, name: $name, goal: $goal})
    RETURN learner.id AS id, learner.name AS name, learner.goal AS goal''', id='hoa', name='Hoa', goal='Data Scientist')
else:
    print('Đổi run_w01_create_demo = True để chạy CREATE.')


Đổi run_w01_create_demo = True để chạy CREATE.


In [ ]:
# W02 — Webhook retry idempotent
run_cypher('''MERGE (learner:Learner:Practice {id: $id})
ON CREATE SET learner.name = $name, learner.goal = $goal
RETURN learner.id AS id, learner.name AS name, learner.goal AS goal''', id='hoa', name='Hoa', goal='Data Scientist')


[{'id': 'hoa', 'name': 'Hoa', 'goal': 'Data Scientist'}]

In [ ]:
# W03 — Update CRM có scope, không tự tạo lead
run_cypher('''MATCH (learner:Learner:Practice {id: $id}) SET learner.background = $background
RETURN learner.id AS id, learner.name AS name, learner.background AS background''', id='hoa', background='Finance')


[{'id': 'hoa', 'name': 'Hoa', 'background': 'Finance'}]

In [ ]:
# W04 — Interest edge idempotent
run_cypher('''MATCH (learner:Learner:Practice {id: $learner_id}) MATCH (course:Course:Practice {id: $course_id})
MERGE (learner)-[:INTERESTED_IN]->(course)
RETURN learner.name AS learner, course.title AS course''', learner_id='hoa', course_id='ai-data-science')


[{'learner': 'Hoa', 'course': 'AI and Data Science'}]

In [ ]:
# W05 — Cleanup test lead có scope. Đặt True sau W01–W04.
run_w05_cleanup = False
if run_w05_cleanup:
    run_cypher("MATCH (learner:Learner:Practice {id: $id}) DETACH DELETE learner", id='hoa')
    print('Deleted only Hoa and Hoa relationships.')
else:
    print('Đổi run_w05_cleanup = True để chạy cleanup.')


Đổi run_w05_cleanup = True để chạy cleanup.


In [ ]:
# W06 — Constraint identity cho Graph RAG ingest
run_cypher("SHOW CONSTRAINTS YIELD name, type, entityType, labelsOrTypes, properties WHERE name = 'practice_entity_identity' RETURN name, type, entityType, labelsOrTypes, properties")


[{'name': 'practice_entity_identity',
  'type': 'UNIQUENESS',
  'entityType': 'NODE',
  'labelsOrTypes': ['Entity'],
  'properties': ['dataset_id', 'key']}]

### Solutions W07–W08 — đọc plan trước, đo traversal sau

W07 dùng `EXPLAIN` để xem lookup theo composite identity. W08 dùng `PROFILE` trên query đã có `dataset_id` ở cả hai endpoint. Chỉ chạy cell này với database bài tập local; đừng copy một PROFILE rộng vào production.

In [ ]:
def plan_summary(query: str, **parameters) -> dict:
    with driver.session(database=NEO4J_DATABASE) as session:
        result = session.run(query, **parameters)
        rows = [record.data() for record in result]
        summary = result.consume()
    plan = summary.plan if query.lstrip().upper().startswith('EXPLAIN') else summary.profile
    if isinstance(plan, dict):
        operator = plan.get('operatorType') or plan.get('operator_type')
        arguments = plan.get('args') or plan.get('arguments') or {}
    else:
        operator = plan.operator_type
        arguments = plan.arguments
    return {'rows': rows, 'root_operator': operator, 'arguments': arguments}

# W07 — EXPLAIN lookup Entity theo composite identity
plan_summary('''
EXPLAIN
MATCH (entity:Entity {dataset_id: $dataset_id, key: $key})
RETURN entity.name AS name, entity.type AS type
''', dataset_id='course-catalog', key='python')


{'rows': [],
 'root_operator': 'ProduceResults@neo4j',
 'arguments': {'planner-impl': 'IDP',
  'Details': 'name, type',
  'planner-version': '5.26.29',
  'string-representation': 'Cypher 5\n\nPlanner COST\n\nRuntime SLOTTED\n\nRuntime version 5.26.29\n\n+----------------------+----+-------------------------------------------------------------------------------------+----------------+\n| Operator             | Id | Details                                                                             | Estimated Rows |\n+----------------------+----+-------------------------------------------------------------------------------------+----------------+\n| +ProduceResults      |  0 | name, type                                                                          |              1 |\n| |                    +----+-------------------------------------------------------------------------------------+----------------+\n| +Projection          |  1 | cache[entity.name] AS name, cache[entity.type]

In [ ]:
# W08 — PROFILE traversal đã dataset-scoped
plan_summary(
    '''
    PROFILE
    MATCH (seed:Entity {dataset_id: $dataset_id, name: $seed_name})
    MATCH path=(seed)-[:RELATED*1..2]-(other:Entity {dataset_id: $dataset_id})
    RETURN DISTINCT other.name AS entity
    ''',
    dataset_id='course-catalog', seed_name='AI and Data Science',
)



{'rows': [{'entity': 'Graph RAG'},
  {'entity': 'Data Analytics Foundations'},
  {'entity': 'Python'},
  {'entity': 'SQL'},
  {'entity': 'Machine Learning'},
  {'entity': 'Deep Learning'},
  {'entity': 'Data Scientist'},
  {'entity': 'Machine Learning Engineer'},
  {'entity': 'Forecasting'}],
 'root_operator': 'ProduceResults@neo4j',
 'arguments': {'GlobalMemory': 2328,
  'planner-impl': 'IDP',
  'Memory': 0,
  'string-representation': 'Cypher 5\n\nPlanner COST\n\nRuntime SLOTTED\n\nRuntime version 5.26.29\n\n+-----------------------------------+----+----------------------------------------------------------------------------+----------------+------+---------+----------------+------------------------+\n| Operator                          | Id | Details                                                                    | Estimated Rows | Rows | DB Hits | Memory (Bytes) | Page Cache Hits/Misses |\n+-----------------------------------+----+-------------------------------------------------

## Solutions G01–G10 — backend retrieval có grounding

In [ ]:
# G01 — Vocabulary allow-list cho entity linker
run_cypher('''
  ''', dataset_id='course-catalog')


[{'name': 'AI and Data Science'},
 {'name': 'Data Analyst'},
 {'name': 'Data Analytics Foundations'},
 {'name': 'Data Scientist'},
 {'name': 'Deep Learning'},
 {'name': 'Forecasting'},
 {'name': 'Graph RAG'},
 {'name': 'Machine Learning'},
 {'name': 'Machine Learning Engineer'},
 {'name': 'Power BI'},
 {'name': 'Python'},
 {'name': 'SQL'}]

In [ ]:
# G02 — Validate seed names không phân biệt casing
run_cypher('''
  ''', dataset_id='course-catalog', seed_names=)


[{'name': 'AI and Data Science',
  'type': 'Course',
  'description': 'A 12-week intermediate AI course'}]

In [48]:
# G03 — RELATED graph facts, distinct edge và citation pointer
run_cypher('''UNWIND $seed_names as seed_name MATCH (entity:Entity)
WHERE toLower(entity.name) = toLower(seed_name)
MATCH path=(entity)-[r:RELATED*1..3]-(other:Entity)

UNWIND relationships(path) as relationship with DISTINCT relationship

return
    startNode(relationship).name as source,
    relationship.type as relationship_type,
    endNode(relationship).name as target

''', seed_names=['AI and Data Science'])



[{'source': 'AI and Data Science',
  'relationship_type': 'REQUIRES',
  'target': 'Python'},
 {'source': 'Data Analytics Foundations',
  'relationship_type': 'TEACHES',
  'target': 'Python'},
 {'source': 'Data Analytics Foundations',
  'relationship_type': 'TEACHES',
  'target': 'SQL'},
 {'source': 'Data Analytics Foundations',
  'relationship_type': 'TEACHES',
  'target': 'Power BI'},
 {'source': 'Data Analytics Foundations',
  'relationship_type': 'LEADS_TO',
  'target': 'Data Analyst'},
 {'source': 'Data Analytics Foundations',
  'relationship_type': 'LEADS_TO',
  'target': 'Data Scientist'},
 {'source': 'Data Analytics Foundations',
  'relationship_type': 'COVERS',
  'target': 'Forecasting'},
 {'source': 'AI and Data Science',
  'relationship_type': 'REQUIRES',
  'target': 'SQL'},
 {'source': 'AI and Data Science',
  'relationship_type': 'TEACHES',
  'target': 'Machine Learning'},
 {'source': 'AI and Data Science',
  'relationship_type': 'TEACHES',
  'target': 'Deep Learning'},
 {'

In [ ]:
# G06 — Fallback source query qua MENTIONS (service chỉ gọi khi G03 chưa có chunk)
run_cypher('''UNWIND $seed_names AS seed_name
MATCH (chunk:Chunk {dataset_id: $dataset_id})-[:MENTIONS]->(entity:Entity {dataset_id: $dataset_id})
WHERE toLower(entity.name) = toLower(seed_name)
RETURN DISTINCT chunk.chunk_id AS chunk_id LIMIT $max_chunks''', dataset_id='course-catalog', seed_names=['AI and Data Science'], max_chunks=8)


[{'chunk_id': 'ai-01'}]

### Graph RAG retrieval: seeds → facts → chunks

Đây là phần sát nhất với `Neo4jGraphStore.retrieve()` của backend 11.2. Entity linker/LLM được giả định đã trả `seed_names`; Cypher không tự hiểu câu hỏi. Nó chỉ bắt đầu từ seed đã được kiểm soát, duyệt `RELATED` tối đa 1–5 hop, trả fact có evidence/chunk ID, rồi tìm source chunks.

Lưu ý `hop_count` được clamp trước khi ghép vào query. Không được ghép input request thô vào syntax `*1..N`. Các giá trị còn lại vẫn parameterized.

In [ ]:
def retrieve_graph_rag_context(
    seed_names: list[str],
    dataset_id: str = 'course-catalog',
    max_hops: int = 3,
    max_facts: int = 40,
    max_chunks: int = 8,
) -> dict:
    hop_count = min(max(int(max_hops), 1), 5)
    fact_query = f'''
    UNWIND $seed_names AS seed_name
    MATCH (seed:Entity {{dataset_id: $dataset_id}})
    WHERE toLower(seed.name) = toLower(seed_name)
    MATCH path=(seed)-[:RELATED*1..{hop_count}]-(other:Entity {{dataset_id: $dataset_id}})
    WHERE other <> seed
    WITH relationships(path) AS rels, seed_name
    UNWIND range(0, size(rels) - 1) AS rel_index
    WITH rels[rel_index] AS rel, rel_index + 1 AS hop_distance, seed_name
    WITH rel, min(hop_distance) AS hop_distance, count(DISTINCT seed_name) AS matched_seed_count
    RETURN startNode(rel).name AS source, rel.type AS relationship,
           endNode(rel).name AS target, rel.evidence AS evidence, rel.chunk_id AS chunk_id,
           hop_distance, matched_seed_count
    ORDER BY hop_distance ASC, matched_seed_count DESC, source ASC, relationship ASC, target ASC
    LIMIT $max_facts
    '''
    facts = run_cypher(
        fact_query, dataset_id=dataset_id, seed_names=seed_names, max_facts=max_facts
    )
    chunk_ids = list(dict.fromkeys(fact['chunk_id'] for fact in facts if fact.get('chunk_id')))[:max_chunks]

    if not chunk_ids:
        fallback_query = '''
        UNWIND range(0, size($seed_names) - 1) AS seed_rank
        WITH $seed_names[seed_rank] AS seed_name, seed_rank
        MATCH (chunk:Chunk {dataset_id: $dataset_id})-[:MENTIONS]->(entity:Entity {dataset_id: $dataset_id})
        WHERE toLower(entity.name) = toLower(seed_name)
        WITH chunk, count(DISTINCT seed_name) AS matched_seed_count, min(seed_rank) AS best_seed_rank
        RETURN chunk.chunk_id AS chunk_id
        ORDER BY matched_seed_count DESC, best_seed_rank ASC, chunk_id ASC
        LIMIT $max_chunks
        '''
        chunk_ids = [row['chunk_id'] for row in run_cypher(
            fallback_query, dataset_id=dataset_id, seed_names=seed_names, max_chunks=max_chunks
        )]

    source_query = '''
    WITH $chunk_ids AS chunk_ids
    WHERE size(chunk_ids) > 0
    UNWIND range(0, size(chunk_ids) - 1) AS chunk_rank
    WITH chunk_ids[chunk_rank] AS chunk_id, chunk_rank
    MATCH (chunk:Chunk {dataset_id: $dataset_id, chunk_id: chunk_id})
    RETURN chunk.chunk_id AS chunk_id, chunk.document_id AS document_id,
           chunk.title AS title, chunk.text AS text, chunk_rank
    ORDER BY chunk_rank ASC
    '''
    sources = run_cypher(source_query, dataset_id=dataset_id, chunk_ids=chunk_ids)
    return {'seed_entities': seed_names, 'facts': facts, 'sources': sources}

retrieval = retrieve_graph_rag_context(['AI and Data Science'], max_hops=3)
retrieval


{'seed_entities': ['AI and Data Science'],
 'facts': [{'source': 'AI and Data Science',
   'relationship': 'COVERS',
   'target': 'Graph RAG',
   'evidence': 'The course covers forecasting and Graph RAG.',
   'chunk_id': 'ai-02'},
  {'source': 'AI and Data Science',
   'relationship': 'REQUIRES',
   'target': 'Python',
   'evidence': 'It requires Python and SQL.',
   'chunk_id': 'ai-01'},
  {'source': 'Data Analytics Foundations',
   'relationship': 'TEACHES',
   'target': 'Python',
   'evidence': 'Students learn Python, SQL, Power BI and ETL for data analysis.',
   'chunk_id': 'analytics-01'},
  {'source': 'AI and Data Science',
   'relationship': 'REQUIRES',
   'target': 'SQL',
   'evidence': 'It requires Python and SQL.',
   'chunk_id': 'ai-01'},
  {'source': 'Data Analytics Foundations',
   'relationship': 'TEACHES',
   'target': 'SQL',
   'evidence': 'Students learn Python, SQL, Power BI and ETL for data analysis.',
   'chunk_id': 'analytics-01'},
  {'source': 'AI and Data Science

In [ ]:
# G04 — Python clamp hop 1–5 trước khi interpolating cấu trúc path
retrieve_graph_rag_context(['AI and Data Science'], dataset_id='course-catalog', max_hops=3, max_facts=40, max_chunks=8)['facts']

[{'source': 'AI and Data Science',
  'relationship': 'COVERS',
  'target': 'Graph RAG',
  'evidence': 'The course covers forecasting and Graph RAG.',
  'chunk_id': 'ai-02'},
 {'source': 'AI and Data Science',
  'relationship': 'REQUIRES',
  'target': 'Python',
  'evidence': 'It requires Python and SQL.',
  'chunk_id': 'ai-01'},
 {'source': 'Data Analytics Foundations',
  'relationship': 'TEACHES',
  'target': 'Python',
  'evidence': 'Students learn Python, SQL, Power BI and ETL for data analysis.',
  'chunk_id': 'analytics-01'},
 {'source': 'AI and Data Science',
  'relationship': 'REQUIRES',
  'target': 'SQL',
  'evidence': 'It requires Python and SQL.',
  'chunk_id': 'ai-01'},
 {'source': 'Data Analytics Foundations',
  'relationship': 'TEACHES',
  'target': 'SQL',
  'evidence': 'Students learn Python, SQL, Power BI and ETL for data analysis.',
  'chunk_id': 'analytics-01'},
 {'source': 'AI and Data Science',
  'relationship': 'LEADS_TO',
  'target': 'Data Scientist',
  'evidence': '

In [ ]:
# G05 — Facts quay về source chunks, không facts × sources
retrieve_graph_rag_context(['AI and Data Science'], dataset_id='course-catalog', max_hops=3, max_facts=40, max_chunks=8)['sources']


[{'chunk_id': 'ai-01',
  'document_id': 'ai-brochure',
  'title': 'AI and Data Science brochure',
  'text': 'AI and Data Science is a 12-week intermediate course. It requires Python and SQL, then teaches Machine Learning and Deep Learning.'},
 {'chunk_id': 'ai-02',
  'document_id': 'ai-brochure',
  'title': 'AI and Data Science brochure',
  'text': 'The course covers forecasting and Graph RAG. It prepares learners for Data Scientist and Machine Learning Engineer roles.'},
 {'chunk_id': 'analytics-01',
  'document_id': 'analytics-brochure',
  'title': 'Data Analytics brochure',
  'text': 'Data Analytics Foundations is a 16-week beginner course. Students learn Python, SQL, Power BI and ETL for data analysis.'},
 {'chunk_id': 'analytics-02',
  'document_id': 'analytics-brochure',
  'title': 'Data Analytics brochure',
  'text': 'Graduates can prepare for Data Analyst and Data Scientist roles. The course includes forecasting projects.'}]

### Solutions G09–G10 — review scope và đối chiếu production

**G09.** `dataset_id` phải xuất hiện ở seed `Entity`, endpoint `Entity` của traversal, `Chunk` trong source lookup, và `Entity` trong fallback `MENTIONS`. Chỉ filter seed vẫn có thể để traversal đi sang Entity khác tenant/dataset; sau đó một fact hoặc source của knowledge base khác lọt vào context. Vì vậy đây là guardrail về correctness và data isolation, không chỉ là tối ưu query.

**G10.** Đọc code production bằng cell dưới. Khi đối chiếu, kiểm tra: name matching của entity linker, clamp hop 1–5 ở Python trước query, fact contract `source/relationship/target/evidence/chunk_id`, và fallback `MENTIONS` chỉ chạy khi facts chưa cho source chunks.

In [ ]:
# G09 — dataset_id phải scope seed, endpoint, source và fallback
dataset_scope_checklist = {
    'seed_entity': 'MATCH (seed:Entity {dataset_id: $dataset_id})',
    'traversal_endpoint': '(other:Entity {dataset_id: $dataset_id})',
    'source_chunk': 'MATCH (chunk:Chunk {dataset_id: $dataset_id, chunk_id: chunk_id})',
    'mentions_fallback': '(chunk:Chunk {dataset_id: $dataset_id})-[:MENTIONS]->(entity:Entity {dataset_id: $dataset_id})',
}
dataset_scope_checklist


{'seed_entity': 'MATCH (seed:Entity {dataset_id: $dataset_id})',
 'traversal_endpoint': '(other:Entity {dataset_id: $dataset_id})',
 'source_chunk': 'MATCH (chunk:Chunk {dataset_id: $dataset_id, chunk_id: chunk_id})',
 'mentions_fallback': '(chunk:Chunk {dataset_id: $dataset_id})-[:MENTIONS]->(entity:Entity {dataset_id: $dataset_id})'}

In [ ]:
def retrieve_as_single_cypher_response(
    seed_names: list[str], dataset_id: str = 'course-catalog', max_hops: int = 3,
    max_facts: int = 40, max_chunks: int = 8,
) -> list[dict]:
    hop_count = min(max(int(max_hops), 1), 5)
    query = f'''
    CALL () {{
      UNWIND $seed_names AS seed_name
      MATCH (seed:Entity {{dataset_id: $dataset_id}})
      WHERE toLower(seed.name) = toLower(seed_name)
      MATCH path=(seed)-[:RELATED*1..{hop_count}]-(other:Entity {{dataset_id: $dataset_id}})
      UNWIND relationships(path) AS rel
      WITH DISTINCT rel
      LIMIT $max_facts
      RETURN collect({{
        source: startNode(rel).name, relationship: rel.type, target: endNode(rel).name,
        evidence: rel.evidence, chunk_id: rel.chunk_id
      }}) AS facts
    }}
    WITH $dataset_id AS dataset_id, facts, reduce(ids = [], fact IN facts |
      CASE WHEN fact.chunk_id IN ids THEN ids ELSE ids + fact.chunk_id END
    )[..$max_chunks] AS chunk_ids
    CALL (chunk_ids, dataset_id) {{
      UNWIND chunk_ids AS chunk_id
      MATCH (chunk:Chunk {{dataset_id: dataset_id, chunk_id: chunk_id}})
      RETURN collect({{
        chunk_id: chunk.chunk_id, document_id: chunk.document_id,
        title: chunk.title, text: chunk.text
      }}) AS sources
    }}
    RETURN facts, sources
    '''
    return run_cypher(
        query, dataset_id=dataset_id, seed_names=seed_names,
        max_facts=max_facts, max_chunks=max_chunks,
    )

retrieve_as_single_cypher_response(['AI and Data Science'])

[{'facts': [{'evidence': 'The course covers forecasting and Graph RAG.',
    'source': 'AI and Data Science',
    'relationship': 'COVERS',
    'chunk_id': 'ai-02',
    'target': 'Graph RAG'},
   {'evidence': 'It requires Python and SQL.',
    'source': 'AI and Data Science',
    'relationship': 'REQUIRES',
    'chunk_id': 'ai-01',
    'target': 'Python'},
   {'evidence': 'Students learn Python, SQL, Power BI and ETL for data analysis.',
    'source': 'Data Analytics Foundations',
    'relationship': 'TEACHES',
    'chunk_id': 'analytics-01',
    'target': 'Python'},
   {'evidence': 'It requires Python and SQL.',
    'source': 'AI and Data Science',
    'relationship': 'REQUIRES',
    'chunk_id': 'ai-01',
    'target': 'SQL'},
   {'evidence': 'Students learn Python, SQL, Power BI and ETL for data analysis.',
    'source': 'Data Analytics Foundations',
    'relationship': 'TEACHES',
    'chunk_id': 'analytics-01',
    'target': 'SQL'},
   {'evidence': 'It prepares learners for Data Scie

In [ ]:
# Chạy cell này khi hoàn tất notebook.
driver.close()
print('Neo4j driver closed.')


Neo4j driver closed.
